# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² structured dataset of genotype–phenotype correlations in non-classical 21-hydroxylase deficiency-associated infertility using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema at:
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL.
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata.
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description from metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema represents three tabular groups (Table 1, 2, 3) as separate record sets. We'll enumerate each record set and the fields in it, referencing everything by their `@id`.


In [ ]:
# List all record set IDs defined in the Croissant schema
record_sets = dataset.record_sets
print("Available Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', rs.get('@id'))}")

# Show all field @ids for each record set
for rs in record_sets:
    print(f"\nFields in Record Set {rs['@id']}:")
    for field in rs['fields']:
        print(f"  - {field['@id']} ({field.get('name', field['@id'])})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. 
All extraction uses only entity `@id`s, following Croissant and FAIR² best practices.

In [ ]:
# Get all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for Record Set {record_set_id}:")
    print(df.columns.tolist())
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing, and grouping. 

For demonstration, we'll use Table 2 (Hormone levels and imaging results):
- RecordSet `@id`: `cr:RecordSet/Table2_HormoneAndImaging`
- Numeric field: choose a hormone level field, e.g. `cr:Field/17OHP_level`.
- Categorical/group field: e.g. imaging result type `cr:Field/Adrenal_CT_Result`.

In [ ]:
# Use Table 2 record set @id
rs_id = 'cr:RecordSet/Table2_HormoneAndImaging'  # Replace with actual @id from schema
df = dataframes.get(rs_id)

# Use hormone field @id for EDA
numeric_field_id = 'cr:Field/17OHP_level'  # Replace with actual @id from schema
if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by adrenal imaging result (categorical @id)
    group_field_id = 'cr:Field/Adrenal_CT_Result'  # Replace with actual @id from schema
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize hormone values using matplotlib and seaborn.

We'll plot hormone distributions from Table 2, referencing every field by its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_field_id = 'cr:Field/17OHP_level'  # As above
group_field_id = 'cr:Field/Adrenal_CT_Result'

# Distribution plot
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in Table 2")
    plt.show()

    # Boxplot by adrenal CT result
    if group_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded FAIR² clinical genotype–phenotype tabulations defined by the Croissant schema, explored available record sets and fields via unique `@id`s, extracted data into DataFrames, filtered and normalized hormone data, and visualized results. Using Croissant ensures transparent referencing and interoperability for biomedical tabular data—facilitating reproducible rare disease analyses. For further investigation, researchers may integrate case features and genetic variants (Table 1 and Table 3) using only their `@id`s, following the FAIR² principles.